### 복습 
- answers, summarys데이터에서 상위의 100개의 데이터를 train 사용 
- 하위의 20개의 데이터를 validation으로 사용 
- DatasetDict를 생성 
- 모델은 'digit82/kobart-summarization' 사용
- tokenizer를 이용 해서 토큰화 인코딩 
- kobart 모델 trainer을 이용하여 반복 학습 
    - input의 최대 사이즈 : 512
    - output의 최대 사이즈 : 256

In [2]:
import pandas as pd 
import numpy as np
from datasets import Dataset, DatasetDict
import evaluate
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
)
from konlpy.tag import Komoran

In [3]:
model_name = 'digit82/kobart-summarization'
# AutoTokenizer가 아닌 Komoran 객체를 생성하는 이유는? -> 
# 실젯값(요약본)과 예측값(텍스트 생성)의 단어들의 일치도를 확인 하기 위함
komoran = Komoran()

In [4]:
df =  pd.read_csv('인터뷰.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12801 entries, 0 to 12800
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   answer   12801 non-null  str  
 1   summary  12801 non-null  str  
dtypes: str(2)
memory usage: 17.0 MB


In [7]:
answers = df['answer'].tolist()
summarys = df['summary'].tolist()

In [8]:
train_docs = answers[:100]
train_sums = summarys[:100]
valid_docs = answers[-20:]
valid_sums = summarys[-20:]

In [9]:
# Dataset으로 이루어진 DatasetDict

raw_ds = DatasetDict(
    {
        'train' : Dataset.from_dict(
            {
                'document' : train_docs, 
                'summary' : train_sums
            }
        ), 
        'validation' : Dataset.from_dict(
            {
                'document' : valid_docs, 
                'summary' : valid_sums
            }
        )
    }
)

In [10]:
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 20
    })
})

In [11]:
DatasetDict(
    {
        'train' : Dataset.from_pandas(df.iloc[:100]), 
        'validation' : Dataset.from_pandas(df.iloc[-20:])
    }
)

DatasetDict({
    train: Dataset({
        features: ['answer', 'summary'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['answer', 'summary'],
        num_rows: 20
    })
})

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 262/262 [00:00<00:00, 65255.80it/s]


In [13]:
# 입출력 길이 설정 
max_input_len = 512
max_target_len = 256

In [21]:
# 토크나이저 함수 
def token_fn(batch):
    # document의 토큰화 
    inputs = tokenizer(
        batch['document'], 
        max_length = max_input_len, 
        padding = 'max_length',                 # 패딩 토큰으로 채워서 길이 유지 
        truncation = True               # max_length보다 텍스트가 길다면 최대 길이 외의 데이터를 제거 
    )

    # with tokenizer.as_target_tokenizer():       # 임시 토크나이저 생성 
    labels = tokenizer(
        batch['summary'], 
        max_length = max_target_len, 
        padding = 'max_length', 
        truncation = True
    )

    labels_ids = np.array(labels['input_ids'])
    # 패딩 토큰의 값들을 -100으로 변환 (손실함수에서 손실 값을 계산 안 하는 기본값 설정)
    labels_ids[labels_ids == tokenizer.pad_token_id] = -100
    inputs['labels'] = labels_ids.tolist()

    return inputs

In [22]:
tokenized_ds = raw_ds.map(
    token_fn, batched = True, remove_columns=['document', 'summary']
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map: 100%|██████████| 20/20 [00:00<00:00, 2093.07 examples/s]


In [23]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer= tokenizer, 
    model = model
)

In [24]:
rouge = evaluate.load('rouge')

In [25]:
# 검증 함수 
def metrics(eval_pred):
    preds, labels = eval_pred

    # labels에서 패딩 토큰의 값들을 다시 tokenizer의 pad_token_id로 변경 
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )
    # 텍스트 디코딩 ( 단어 사전의 id에 대응하는 문자로 변환 )
    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 디코딩 과정에서 좌우의 공백이 존재하는 경우에는 측정값이 달라질수 있으므로 공백을 제거 
    pred_str = [doc.strip() for doc in pred_str]
    label_str = [doc.strip() for doc in label_str]

    # rouge 계산
    result = rouge.compute(
        predictions= pred_str, 
        references= label_str, 
        tokenizer = lambda x : komoran.morphs(x)
    )

    # rouge를 보기 편한 형태로 변환 
    result = {k : round(v * 100, 2) for k, v in result.items()}

    return result

In [26]:
args = Seq2SeqTrainingArguments(
    output_dir= './kobart', 
    eval_strategy= 'epoch', 
    save_strategy='epoch', 
    learning_rate= 5e-05, 
    num_train_epochs=5, 
    logging_steps=1, 

    predict_with_generate=True, 
    generation_max_length=64, 
    generation_num_beams=4, 

    load_best_model_at_end=True, 
    metric_for_best_model= 'rougeL', 
    greater_is_better=True, 
    dataloader_num_workers=3
)

In [27]:
trainer = Seq2SeqTrainer(
    model = model, 
    args = args,
    train_dataset= tokenized_ds['train'], 
    eval_dataset= tokenized_ds['validation'], 
    processing_class= tokenizer, 
    data_collator= data_collator, 
    compute_metrics= metrics
)

In [28]:
# 파인 튜닝 
trainer.train()

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.115796,1.088037,58.950000,39.480000,43.350000,43.820000
2,0.685070,1.118847,59.380000,40.290000,43.200000,43.030000
3,0.690680,1.145350,59.250000,39.540000,41.940000,42.090000
4,0.310200,1.230336,60.620000,41.810000,42.960000,43.240000
5,0.314905,1.248097,60.840000,41.910000,44.350000,44.490000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.66it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.45it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.65it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100

TrainOutput(global_step=65, training_loss=0.6459665449766012, metrics={'train_runtime': 958.7343, 'train_samples_per_second': 0.522, 'train_steps_per_second': 0.068, 'total_flos': 152434114560000.0, 'train_loss': 0.6459665449766012, 'epoch': 5.0})

In [51]:
# 예측 -> answers의 임의의 데이터를 선택 -> 요약본 예측 텍스트 생성 
test_text = answers[200]
# test_text = answers
# test_text

In [52]:
inputs = tokenizer(
    test_text, 
    return_tensors = 'pt', 
    truncation = True, 
    max_length = max_input_len, 
    padding = 'max_length'
)

In [53]:
inputs

{'input_ids': tensor([[14323, 16530, 14446, 24407, 18964, 22424, 14385, 20658, 12926, 14174,
         11440, 17150, 17216, 16896, 22776, 15332, 14543, 15995, 24407, 17291,
         21234, 14147, 15995, 17508, 23235, 15761, 14315, 19737, 16891, 20865,
         17508, 22557,  9754, 17150, 16393, 14605, 17498, 19126, 14487, 14641,
         25154, 16218, 14051, 14790, 17698, 11465, 29147, 14955, 14339, 14930,
         20910, 14032, 14082, 28822, 16313, 12332, 20364, 17150, 14246, 14030,
         14025, 17216, 15350,  9049, 15703,  9102, 14487, 14025, 14427, 14030,
         20003, 14262, 15995, 15416, 14160, 14108, 21548, 17053, 17131, 17939,
         21194, 16145, 14339, 16707, 14056, 15706, 12264, 28326, 17150, 20244,
         29629, 14955, 15416, 17939, 15709, 14593, 20423, 24067, 14646, 13594,
         14032, 16846, 16410, 17150, 16393, 17498,  9102, 16730, 14467, 14142,
         14282, 16218, 13590, 14376, 17875, 20628, 14955, 21412, 29989, 14056,
         14641, 17634, 16010, 14172, 1

In [48]:
gen_ids = model.generate(
    **inputs.to(model.device), 
    max_new_tokens = 500, 
    min_new_tokens = 10, 
    num_beams = 4, 
    do_sample = True, 
    length_penalty = 0.6, 
    no_repeat_ngram_size = 2, 
    repetition_penalty = 3.0, 
    eos_token_id = tokenizer.eos_token_id, 
    pad_token_id = tokenizer.pad_token_id
)

KeyboardInterrupt: 

In [42]:
gen_ids

tensor([[    0, 16530, 14446, 18964, 22424, 14385, 20658, 12926, 14174, 11440,
         17150, 17216, 15350,  9049, 15703,  9102, 14025, 14427, 14030, 20003,
         14262, 15995, 15416, 14160, 14108, 21548, 17053, 17131, 17939, 21194,
         14339, 16707, 14056, 15706, 12264, 16062, 15382, 11820, 15170, 14955,
         14355, 16311, 16010, 21051, 14326, 11786, 17279, 21941,  9049, 14464,
         14033, 14163, 15170,   233,     1]])

In [44]:
print(tokenizer.decode(gen_ids[0], skip_special_tokens=True))

저는 지금연이 지은 크리티컬 매스 라는 책을 읽게 되었고 이 책에서 느낀 게 제가 정말 남들이 감동시킬 만한 노력을 한다면 타인들이 다 알아줄 것이라는 이야기였습니다. 그래서 다른 사람의 이야기를 듣는 법에 대해서도 깨닫게 된 것 같습니다."


In [45]:
summarys[200]

'백지연의 크리티컬 매스 라는 책을 요즘 읽고 있는데, 소통 능력이 굉장히 뛰어나고 다양한 사람들과 인터뷰 경력이 있잖아요. 그래서 타인을 이해할 수 있는 폭이 넓지 않을까 라는 생각을 했습니다. 이 책을 읽으면서 인터뷰를 하고 다른 사람의 이야기를 듣는 법에 대해서도 깨달은 것 같습니다.'

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12801 entries, 0 to 12800
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   answer   12801 non-null  str  
 1   summary  12801 non-null  str  
dtypes: str(2)
memory usage: 17.0 MB


In [50]:
model2 = AutoModelForSeq2SeqLM.from_pretrained("./kobart_exam")

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 5705.19it/s]


In [54]:
gen_ids = model2.generate(
    **inputs.to(model.device), 
    max_new_tokens = 500, 
    min_new_tokens = 10, 
    num_beams = 4, 
    do_sample = True, 
    length_penalty = 0.6, 
    no_repeat_ngram_size = 2, 
    repetition_penalty = 3.0, 
    eos_token_id = tokenizer.eos_token_id, 
    pad_token_id = tokenizer.pad_token_id
)
print(tokenizer.decode(gen_ids[0], skip_special_tokens=True))

백지연의 소통 능력이 굉장히 뛰어나다는 생각이 들었고, 다양한 사람들과 인터뷰 경력이 있으셔서 타인을 이해할 수 있는 폭이 넓지 않을까 라는 생각을 했습니다. 그래서 이 책을 읽으면서 인터뷰를 하고 다른 사람의 이야기를 듣는 법에 대해서도 깨닫게 된 것 같습니다. 인터뷰를 하면서 다른 사람들의 이야기를 들을 때에도 중요하다는 것을 깨닫는 것 같았습니다.  제가 감명 깊어졌습니다. 그리고 인터뷰를 하는 방법도 깨달았습니다.
